In [2]:
import faiss
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load product metadata (50K)
products_df = pd.read_csv("data/products_50k.csv")

# Load FAISS index
index = faiss.read_index("data/faiss_index_50k.bin")

# Load embedding model (same as Notebook 4)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("LLM Agent Notebook Initialized.")


LLM Agent Notebook Initialized.


In [3]:
ranking_model_name = "Qwen/Qwen2.5-0.5B-Instruct"

ranking_tokenizer = AutoTokenizer.from_pretrained(ranking_model_name)

ranking_model = AutoModelForCausalLM.from_pretrained(
    ranking_model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

device = "cuda" if torch.cuda.is_available() else "cpu"
ranking_model = ranking_model.to(device)

print("Ranking Agent Model Loaded:", ranking_model_name, "on", device)


`torch_dtype` is deprecated! Use `dtype` instead!


Ranking Agent Model Loaded: Qwen/Qwen2.5-0.5B-Instruct on cuda


In [4]:
def ranking_agent(query_text, top_k=5):
    # Step 1: Embed the query text
    query_vec = embedder.encode([query_text], convert_to_numpy=True)
    query_vec = query_vec.astype("float32")
    faiss.normalize_L2(query_vec)
    
    # Step 2: Retrieve FAISS top-K
    distances, indices = index.search(query_vec, top_k)
    candidate_rows = products_df.iloc[indices[0]]
    
    # Build context text for LLM
    candidate_descriptions = ""
    for i, row in enumerate(candidate_rows.itertuples()):
        candidate_descriptions += f"[{i}] Title: {row.title}\nDescription: {row.description}\n\n"
    
    # Step 3: Ranking prompt for Qwen
    prompt = f"""
You are a product ranking agent. Rank the following {top_k} products by how well they match the user's query.

User query: "{query_text}"

Products:
{candidate_descriptions}

Return ONLY a Python list of indices in best-to-worst order. Example: [2, 0, 1, 3, 4]
    """
    
    # Tokenize prompt
    inputs = ranking_tokenizer(prompt, return_tensors="pt").to(device)
    
    # Generate LLM output
    outputs = ranking_model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )
    
    result = ranking_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Step 4: Extract index list from output
    import re
    match = re.search(r"\[(.*?)\]", result)
    if match:
        order = [int(x.strip()) for x in match.group(1).split(",")]
    else:
        order = list(range(top_k))  # fallback
    
    # Return ranked rows
    return candidate_rows.iloc[order], result


In [5]:
test_query = "I need a durable sports water bottle for outdoor hiking."

ranked_df, llm_output = ranking_agent(test_query, top_k=5)

print("LLM Output:")
print(llm_output)
print("\nRanked Products:")
ranked_df[["item_id", "title", "description"]]


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LLM Output:

You are a product ranking agent. Rank the following 5 products by how well they match the user's query.

User query: "I need a durable sports water bottle for outdoor hiking."

Products:
[0] Title: sharpro 32 oz Leakproof BPA Free Drinking Sports Water Bottle with Motivational Time Marker & Straw to Enough Water Drinking For The Day Fitness and Outdoor Enthusiasts (A6
Description: Yellow/Blue Gradient). STAY HYDRATED ALL THE TIME WITH THE MOST INNOVATIVE WATER BOTTLE BY FIDUS! Drinking enough water is easier said than done for some people,and these water bottles are here to help you out.Having a water bottle at your desk or in your gym bag is one thing, but remembering to fill it up constantly is another. One helpful tactic comes in the form of these time-labeled water bottles, which make it easy to see how much water you're really drinking as the time passes and when it's time to get a refill ASAP. And Fidus' innovative 32oz water bottles with strap & time marker are just

,item_id,title,description
5067,B09JBSZRK5,sharpro 32 oz Leakproof BPA Free Drinking Spor...,Yellow/Blue Gradient). STAY HYDRATED ALL THE T...


In [6]:
def pricing_agent(product_row, base_price=None):
    """
    product_row: a pandas row from products_sample (after retrieval/ranking)
    base_price: optional override; if None → use extracted price
    """

    title = product_row.title
    desc = product_row.description
    price = base_price if base_price is not None else product_row.price
    segment = product_row.price_segment

    # Build prompt
    prompt = f"""
You are a pricing analysis agent. Analyze whether the product is fairly priced for typical consumers.

Product Title: {title}
Description: {desc}
Listed Price: {price}
Price Segment: {segment}

Based on similar products, quality indicators, and general consumer expectations, answer:
1. Is the price fair, too high, or a good value?
2. How would an average user rate the price–quality balance?
3. What price range would you recommend?
4. Give a short explanation.

Return your answer in a clean JSON format with keys:
- fairness
- expected_rating
- recommended_range
- explanation
"""

    inputs = ranking_tokenizer(prompt, return_tensors="pt").to(device)

    output = ranking_model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

    response = ranking_tokenizer.decode(output[0], skip_special_tokens=True)

    return response


In [9]:
from google.cloud import bigquery
client = bigquery.Client()


In [10]:
dim_product = client.query("""
    SELECT item_id, price, price_segment
    FROM `linear-theater-436300-r9.ecommerce_pipeline.dim_product`
""").to_dataframe()

products_df = products_df.merge(dim_product, on="item_id", how="left")

print(products_df.head())
print(products_df.columns)


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


      item_id                                              title  \
0  B00KQ1J568  Houseoftoners Remanufactured Ink Cartridge Rep...   
1  B005FA2ZVW  Extraordinary Images of Jesus Christ -From the...   
2  B07H9DPQVQ  Biocorrect© Comfort-FIT Men's/Women's/Youth - ...   
3  B0BW1YLTJN            Fate Worse than Death (Octavia Hollows)   
4  B01IW417X4  Clara Clark Best Travel Neck Pillow & Eye Shad...   

                                         description  \
0  2-PACK Remanufactured inkjet cartridge for: 2-...   
1                                                  .   
2                                                  .   
3                                                  .   
4                                                  .   

                                  text_for_embedding  price price_segment  
0  Houseoftoners Remanufactured Ink Cartridge Rep...    2.0           low  
1  Extraordinary Images of Jesus Christ -From the...    NaN       premium  
2  Biocorrect© Comfort-FIT

In [15]:
# Re-run ranking with the same test query
test_query = "I need a durable sports water bottle for outdoor hiking."
ranked_df, llm_output = ranking_agent(test_query, top_k=5)

# Now take the new top-ranked product
top_product = ranked_df.iloc[0]

# Run pricing agent
pricing_result = pricing_agent(top_product)

print("Pricing Agent Output:\n")
print(pricing_result)


Pricing Agent Output:


You are a pricing analysis agent. Analyze whether the product is fairly priced for typical consumers.

Product Title: sharpro 32 oz Leakproof BPA Free Drinking Sports Water Bottle with Motivational Time Marker & Straw to Enough Water Drinking For The Day Fitness and Outdoor Enthusiasts (A6
Description: Yellow/Blue Gradient). STAY HYDRATED ALL THE TIME WITH THE MOST INNOVATIVE WATER BOTTLE BY FIDUS! Drinking enough water is easier said than done for some people,and these water bottles are here to help you out.Having a water bottle at your desk or in your gym bag is one thing, but remembering to fill it up constantly is another. One helpful tactic comes in the form of these time-labeled water bottles, which make it easy to see how much water you're really drinking as the time passes and when it's time to get a refill ASAP. And Fidus' innovative 32oz water bottles with strap & time marker are just the thing you're looking for! Benefit from Numerous Intuitively Desi

In [16]:
def orchestrator_agent(user_query, top_k=5):
    """
    Main entry point for the multi-agent system.
    Steps:
    1. Embed + retrieve with FAISS
    2. Rank with LLM (ranking_agent)
    3. Price analysis with pricing_agent
    4. Return final JSON-style response
    """

    # 1. Ranking Agent → top candidates
    ranked_df, ranking_llm_output = ranking_agent(user_query, top_k=top_k)

    # top product
    best_product = ranked_df.iloc[0]

    # 2. Pricing Agent → price reasoning
    pricing_output = pricing_agent(best_product)

    # 3. Build final structured output
    final_result = {
        "query": user_query,
        "recommended_item": {
            "item_id": best_product["item_id"],
            "title": best_product["title"],
            "description": best_product["description"],
            "price": best_product.get("price", None),
            "price_segment": best_product.get("price_segment", None)
        },
        "ranking_agent_output": ranking_llm_output,
        "pricing_agent_output": pricing_output
    }

    return final_result


In [ ]:
test_query = "I'm looking for a durable, leakproof water bottle for hiking and outdoor sports."

result = orchestrator_agent(test_query, top_k=5)

import json
print(json.dumps(result, indent=2))


{
  "query": "I'm looking for a durable, leakproof water bottle for hiking and outdoor sports.",
  "recommended_item": {
    "item_id": "B09JBSZRK5",
    "title": "sharpro 32 oz Leakproof BPA Free Drinking Sports Water Bottle with Motivational Time Marker & Straw to Enough Water Drinking For The Day Fitness and Outdoor Enthusiasts (A6",
    "description": "Yellow/Blue Gradient). STAY HYDRATED ALL THE TIME WITH THE MOST INNOVATIVE WATER BOTTLE BY FIDUS! Drinking enough water is easier said than done for some people,and these water bottles are here to help you out.Having a water bottle at your desk or in your gym bag is one thing, but remembering to fill it up constantly is another. One helpful tactic comes in the form of these time-labeled water bottles, which make it easy to see how much water you're really drinking as the time passes and when it's time to get a refill ASAP. And Fidus' innovative 32oz water bottles with strap & time marker are just the thing you're looking for! Benefit

: 